# 01 · 배포 버전과 실행 설정

목표: 같은 코드의 설정 차이를 식별하고 시작·준비·종료 상태를 확인한다. 약12분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client

## 기본 설정

릴리스·프롬프트·모델·데이터가 따로 기록되는 이유를 설명해본다.

In [ ]:
cfg = Settings()
manifest = cfg.manifest()
manifest

## 설정의 동일성

같은 입력은 같은 config_id다. 코드 commit이나 이미지 digest를 대신하는 값은 아니다.

In [ ]:
assert manifest == Settings().manifest()
changed = replace(cfg, timeout_s=0.5)
assert changed.manifest()["config_id"] != manifest["config_id"]
changed.manifest()

## 시작 전 실패

오타·NaN·과도한 처리량을 조용히 수용하지 않는다.

In [ ]:
for values in [{"release": "latest"}, {"timeout_s": float("nan")}, {"capacity": 0}]:
    try:
        Settings(**values)
    except ValueError as e:
        print(type(e).__name__, str(e))
    else:
        raise AssertionError("invalid configuration accepted")

## 앱의 수명주기

ASGITransport는 startup을 자동 실행하지 않는다. lab_client가 lifespan을 명시적으로 연다.

In [ ]:
from week10.lab.app import create_app

app = create_app(cfg)
assert not app.state.started
async with lab_client(cfg) as (running, client):
    print((await client.get("/health/live")).json())
    assert (await client.get("/health/ready")).status_code == 200
assert not running.state.started and running.state.draining

## 버전 응답

요청 결과의 release와 서버 manifest를 대조한다.

In [ ]:
async with lab_client(cfg) as (app, client):
    r = await client.post("/api/v1/assist", json={"message": "신청 절차"})
    assert r.headers["X-Release"] == (await client.get("/release")).json()["release"]
    print(r.json())

## 컨테이너 실행 경로

파일을 읽기만 한다. Docker 설치·다운로드·데몬 실행을 자동 수행하지 않는다.

In [ ]:
dockerfile = (ROOT / "week10/lab/Dockerfile").read_text()
print(dockerfile)
assert "USER 10001" in dockerfile and "--host" in dockerfile

## 내 설정의 경계

동시 처리량을 1·2로 바꿔 manifest 차이를 확인한다. 다음 장에서 실제 거절 수를 관찰한다.

In [ ]:
for cap in [1, 2]:
    print(Settings(capacity=cap).manifest())

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.